# Word2Vec

In [ ]:
V = 10_000
D = 100

print(f"Number of parameters with the hidden layer: {2 * V * D + D + V:,}")
print(f"Number of parameters of the logistic regression model: {V * V + V:,}")

$$
\hat{y} = \text{softmax}(z)
$$

$$
l = \sum_{j=1}^{V} y_{j} \log(\hat{y}_{j})
$$

## Example

In [ ]:
import numpy as np

V = 4
D = 2

np.random.seed(32)

Wh = np.random.randn(D, V)
Wo = np.random.randn(V, D)

print("Wh:")
print(Wh)

print("\nWo:")
print(Wo)
print("\nWh shape:", Wh.shape)
print("Wo shape:", Wo.shape)

In [ ]:
# For the first word (Alice, at this case index 0 in the vocabulary in numpy)

x_alice = np.zeros(V)
x_alice[0] = 1

print("\nx_alice:")
print(x_alice)

In [ ]:
h = Wh @ x_alice

print("\nh:")
print(h)

z = Wo.dot(h)
print("\nz:")
print(z)

y_hat = np.exp(z) / np.sum(np.exp(z))
print("\ny_hat:")
print(y_hat)

print("\nThe actual word following 'Alice' is 'was', which is index 4 in the vocabulary (3 in numpy).")

y_was = np.zeros(V)
y_was[3] = 1

print("\ny_was:")
print(y_was)


## Gradient Descent Updates

$$
\delta = \underbrace{\hat{y} - y}_{V \times 1}
$$

$$
\underbrace{\frac{\partial L}{\partial W_{o}}}_{V \times D} = \underbrace{(\hat{y} - y)}_{V \times 1} \, \underbrace{h^{\top}}_{1 \times D}
$$

$$
W_{o} \leftarrow W_{o} - \eta \, \frac{\partial L}{\partial W_{o}} = W_{o} - \eta \, (\hat{y} - y) \, h^{\top}.
$$

$$
\underbrace{\frac{\partial L}{\partial W_{h}}}_{D \times V} = \left(\underbrace{W_{o}^{\top}}_{D \times V} \underbrace{(\hat{y} - y)}_{V \times 1}\right) \, \underbrace{x^{\top}}_{1 \times V}.
$$

$$
W_{h} \leftarrow W_{h} - \eta \, \frac{\partial L}{\partial W_{h}} = W_{h} - \eta \, \left(W_{o}^{\top} (\hat{y} - y)\right) \, x^{\top}.
$$

## The Skip-Gram Model

In [ ]:
from nltk.tokenize import word_tokenize

sentence = "They are simply cheques that men draw on a bank where they have no account".lower()
tokens = word_tokenize(sentence)

window_size = 2
skipgram_samples = []

for idx, center_word in enumerate(tokens):
    print(f"\nCenter word: '{center_word}' (index {idx})")
    
    for offset in range(-window_size, window_size + 1):
        context_idx = idx + offset
        
        if offset == 0 or context_idx < 0 or context_idx >= len(tokens):
            continue
        context_word = tokens[context_idx]

        print(f"  Context word: '{context_word}' (index {context_idx})")
        skipgram_samples.append((center_word, context_word))

print("Tokens:", tokens)
print("Skip-gram samples (center, context):")
for sample in skipgram_samples:
    print(sample)

## Negative Sampling

$$
\hat{y} = \sigma (w^{o}_{c} \cdot w_{i}) = \frac{1}{1 + \exp(-w^{o}_{c} \cdot w_{i})}
$$

$$
L = - \left( \sum_{(w_i, w_c) \in D^{+}} \log(\hat{y}) + \sum_{(w_i, w_c) \in D^{-}} \log(1 - \hat{y}) \right)
$$

## Choosing Negative Samples

## Subsampling Frequent Words

In [ ]:
import numpy as np
import nltk

nltk.download("gutenberg")
from nltk.corpus import gutenberg
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:  # e.g. Google Colab
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
gutenberg.fileids()
alice = gutenberg.raw(fileids="carroll-alice.txt")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
words = word_tokenize(alice)
words = [word.lower() for word in words if word.isalpha()]

words_freq_dist = FreqDist(words)
words_freq_dist.most_common(20)

## Gradient Descent Updates with Negative Sampling

$$
\underbrace{\frac{\partial L}{\partial w^{o}_{c}}}_{D \times 1} = \frac{\partial L}{\partial a} \frac{\partial a}{\partial w^{o}_{c}} = \underbrace{(\hat{y} - y)}_{\text{scalar}} \, \underbrace{w_{i}}_{D \times 1}
$$

$$
w^{o}_{c} \leftarrow w^{o}_{c} - \eta \, \frac{\partial L}{\partial w^{o}_{c}} = w^{o}_{c} - \eta \, (\hat{y} - y) \, w_{i}.
$$

$$
\underbrace{\frac{\partial L}{\partial w_{i}}}_{D \times 1} = \frac{\partial L}{\partial a} \frac{\partial a}{\partial w_{i}} = \underbrace{(\hat{y} - y)}_{\text{scalar}} \, \underbrace{w^{o}_{c}}_{D \times 1}
$$

## Implementation in Numpy

### Preparing the Data

In [ ]:
import nltk
nltk.download("gutenberg")

from nltk.corpus import gutenberg

# Read the full text of "Alice in Wonderland" from the Gutenberg corpus
alice_text = gutenberg.raw(fileids="carroll-alice.txt")
print(alice_text[:300])  # Print the first 1000 characters as a preview

In [ ]:
# Get a list of sentences and their tokens

from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(alice_text)

# A list (sentences) of lists (tokens in a single sentence) of tokens
tokenized_sentences = [
    [token for token in word_tokenize(sentence.lower()) if token.isalpha()] for sentence in sentences
    ]

# Print the first few sentences as preview

for sent_tokens in tokenized_sentences[:2]:
    print("#" * 20)
    for token in sent_tokens:
        print(token)
    
all_tokens = [token for sentence in tokenized_sentences for token in sentence if token.isalpha()]

In [ ]:
# Count unique extracted tokens, this is the vocabulary size

V = len(set(all_tokens))

print(f"Total tokens: {len(all_tokens)}")
print(f"Vocabulary size: {V}")

In [ ]:
# After that, compute the word frequencies

from nltk.probability import FreqDist

word_freq_dist = FreqDist(all_tokens)
word_freq_dist.most_common(20)

In [ ]:
# Create a mapping from word to index and vice versa

word2idx = {word: idx for idx, (word, _) in enumerate(word_freq_dist.items())}
idx2word = {idx: word for word, idx in word2idx.items()}

print("Index of alice:", word2idx.get("alice", "Not found"))
print("Index of wonderland:", word2idx.get("wonderland", "Not found"))

In [ ]:
# Put it in a DataFrame for better handling
import pandas as pd

word_freq_df = pd.DataFrame(word_freq_dist.items(), columns=['word', 'frequency']).sort_values(by='frequency', ascending=False)
word_freq_df.head(20)

In [ ]:
# Create the subsampling probabilities

word_freq_df["p"] = word_freq_df["frequency"] / word_freq_df["frequency"].sum()
word_freq_df["p1"] = word_freq_df["p"]**(3/4)
word_freq_df["p1"] = word_freq_df["p1"] / word_freq_df["p1"].sum()

word_freq_df["word_idx"] = word_freq_df["word"].map(word2idx)
word_freq_df = word_freq_df.set_index("word_idx").sort_index()

drop_threshold = 1e-4

word_freq_df["p_drop"] = 1 - np.sqrt(drop_threshold / word_freq_df["p1"])

word_drop_p = word_freq_df["p_drop"].to_dict()
word_freq_df

In [ ]:
# A helper function to select negative samples based on the unigram distribution

def get_negative_samples(n):
    neg_words = word_freq_df["word"].sample(
        n=n,
        replace=True,
        weights=word_freq_df["p1"]
    )

    return [word2idx[word] for word in neg_words]


get_negative_samples(5)

In [ ]:
# Create the training pairs from the full text. Each pair is (center_word_idx, context_word_idx, positive/negative label)

training_pairs = []

for sentence in tokenized_sentences:
    sentence_length = len(sentence)
    
    # Skip too short sentences
    if sentence_length < 2:
        continue

    # Apply subsampling
    
    training_tokens = []

    for word in sentence:
        random_value_between_0_1 = np.random.random()
        drop_prob = word_drop_p.get(word2idx.get(word, -1), 0.0)
        if random_value_between_0_1 > drop_prob:
            training_tokens.append(word)
    

    sentence = [word for word in training_tokens]
    sentence_length = len(training_tokens)
        
    for center_pos, center_word in enumerate(training_tokens):
        center_idx = word2idx[center_word]
        
        # Define context window
        window_start = max(0, center_pos - 2)
        window_end = min(sentence_length, center_pos + 3)
        
        for context_pos in range(window_start, window_end):
            if context_pos == center_pos:
                continue
            
            context_word = sentence[context_pos]
            if context_word not in word2idx:
                continue
            
            context_idx = word2idx[context_word]
            
            # Positive sample
            training_pairs.append((center_idx, context_idx, 1))
            
            # Negative samples (should fix)
            negative_samples = get_negative_samples(4)
            for neg_idx in negative_samples:
                training_pairs.append((center_idx, neg_idx, 0))

In [ ]:
# Now we are ready to write the training loop

# First we initialize the embedding matrices

Wh = np.random.randn(D, V) * 0.01
Wo = np.random.randn(V, D) * 0.01

# Then we say how many times we want to go through the full text
epochs = 5

# Also set the learning rate
learning_rate = 0.01

for epoch in range(epochs):
    # Shuffle training pairs each epoch so that the words are not always in the same order
    np.random.shuffle(training_pairs)
        
    total_loss = 0
    for center_idx, context_idx, y in training_pairs:
        # Forward pass: compute predicted y_hat

        w_i = Wh[:, center_idx]  # Hidden layer
        z = Wo[context_idx, :].dot(w_i)  # Output layer

        y_hat = 1 / (1 + np.exp(-z))  # Sigmoid: predicted probability
        loss = - (y * np.log(y_hat + 1e-10) + (1 - y) * np.log(1 - y_hat + 1e-10))  # Loss 

        # Record the total loss for printing
        total_loss += loss       

        # Backward pass: update the weights

        e = y_hat - y  # Error term
        Wo[context_idx, :] -= learning_rate * e * w_i  # Update output weights
        Wh[:, center_idx] -= learning_rate * e * Wo[context_idx, :]  # Update input weights

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(training_pairs):.4f}")

## Technical Appendix

### Derivation of Gradient Descent Updates in the Bi-gram Model

#### Output Layer Gradient

#### Hidden Layer Gradient

## Derivation of Gradient Descent Updates (Negative Sampling)